In [2]:
import os,sys
import pandas as pd
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(root)
from utils import utils,models

In [2]:
dataset_dir = '../dataset/newsimages_test_and_evaluation_26_v1.0'
images_dir = f'{dataset_dir}/news_images_evaluation'
dataset = pd.read_csv(f"{dataset_dir}/news_articles_combined.csv",encoding="utf-8-sig")
dataset.set_index('article_id', inplace=True)
dataset['article_title'] = dataset['article_title'].apply(utils.normalize_text)

In [3]:
generator = models.OllamaPipeline()

In [4]:
dataset.columns

Index(['article_url', 'article_title', 'image_id'], dtype='str')

In [ ]:
from tqdm import tqdm

results = []
checkpoint_path = "../dataset/qe_results.json"

try:
    results = utils.load_json(checkpoint_path)
    start_idx = len(results)
    print(f"Resuming from {start_idx}")
except:
    results = []
    start_idx = 0

for i, row in enumerate(tqdm(dataset.itertuples(), total=len(dataset), desc="QE Generation")):
    
    if i < start_idx:
        continue  # skip already processed rows

    query = row.article_title
    expanded = utils.clean_prompt(utils.generate_expansion(generator, query))
    
    results.append({
        "query": query,
        "expanded": expanded,
        "id": row.Index
    })

    # Save checkpoint every 100 iterations
    if (i + 1) % 100 == 0:
        utils.save_json(results, checkpoint_path)

# Final save (important)
utils.save_json(results, checkpoint_path)

Resuming from 15180


QE Generation: 100%|██████████| 15180/15180 [00:00<00:00, 552984.55it/s]


In [3]:
checkpoint_path = "../dataset/qe_results.json"
cached_qe = utils.load_json(checkpoint_path)

In [ ]:
cached_qe_dict = {
    query['id']:{'article_title':query['query'],'expanded':query['expanded'] } for query in cached_qe
}
utils.save_json(cached_qe_dict,"../dataset/qe_results_dictionary.json")

In [4]:
title_expanded_map = {query['query']:query['expanded'] for query in cached_qe} 

In [6]:
utils.save_json(title_expanded_map,"../dataset/title_to_expanded_mapping.json")